In [ ]:
# Importando bibliotecas
import re                                   # regex
import unicodedata                          # normalização de texto
import pandas as pd                         # dataframes
from pathlib import Path                    # caminhos de arquivo
from openpyxl import Workbook               # criar excel
from openpyxl.styles import Font            # fonte
from openpyxl.styles import PatternFill     # cor de fundo
from openpyxl.styles import Border          # borda
from openpyxl.styles import Side            # lado da borda
from openpyxl.styles import Alignment       # alinhamento
from openpyxl.utils import get_column_letter  # letra da coluna

In [ ]:
# =========================================================
# config
# =========================================================
arquivo_parquet = "df_ronda.parquet"

data_inicio = "2024-01-01"
data_fim = "2026-01-31"

limite_acima = pd.Timedelta(hours=10, minutes=50)
limite_abaixo = pd.Timedelta(hours=8)

# se você tiver a lista das 81 chapas, coloque aqui (strings)
chapas_pinese = None

# False = usa todas as batidas E/S do dia
# True  = usa apenas coletores com "port"
filtrar_somente_portaria = False

out_dir = Path("saida_relatorios_ronda")
out_dir.mkdir(exist_ok=True)

# =========================================================
# helpers
# =========================================================
def normalizar_texto(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).strip()
    valor = "".join(
        ch for ch in unicodedata.normalize("NFKD", valor)
        if not unicodedata.combining(ch)
    )
    valor = re.sub(r"\s+", " ", valor)
    return valor

def formatar_timedelta(td):
    if pd.isna(td):
        return None
    total_segundos = int(td.total_seconds())
    if total_segundos < 0:
        return None
    horas = total_segundos // 3600
    minutos = (total_segundos % 3600) // 60
    return f"{horas:02d}:{minutos:02d}"

def dia_semana_ptbr(data_valor):
    dias = {
        0: "segunda-feira",
        1: "terca-feira",
        2: "quarta-feira",
        3: "quinta-feira",
        4: "sexta-feira",
        5: "sabado",
        6: "domingo",
    }
    return dias.get(pd.Timestamp(data_valor).weekday())

def somar_intervalos_entrada_saida(grupo):
    """
    soma intervalos entrada -> saida no dia
    - ignora saida sem entrada anterior
    - se houver varias entradas seguidas, mantém a primeira
    - se houver varias saidas seguidas, mantém a ultima
    - se sobrar entrada sem saida, marca inconsistencia
    """
    grupo = grupo.sort_values("data_marcacao").copy()

    eventos = []
    ultimo_mov = None
    qtd_saidas_sem_entrada = 0

    for _, row in grupo.iterrows():
        mov = row["movimento"]
        ts = row["data_marcacao"]

        if mov not in ["entrada", "saida"]:
            continue

        if not eventos:
            if mov == "entrada":
                eventos.append((mov, ts))
                ultimo_mov = mov
            else:
                qtd_saidas_sem_entrada += 1
            continue

        if mov == ultimo_mov:
            if mov == "entrada":
                continue
            else:
                eventos[-1] = (mov, ts)
                continue

        eventos.append((mov, ts))
        ultimo_mov = mov

    total = pd.Timedelta(0)
    qtd_intervalos = 0
    entrada_aberta = None

    for mov, ts in eventos:
        if mov == "entrada":
            if entrada_aberta is None:
                entrada_aberta = ts

        elif mov == "saida":
            if entrada_aberta is not None and ts >= entrada_aberta:
                total += (ts - entrada_aberta)
                qtd_intervalos += 1
                entrada_aberta = None
            else:
                qtd_saidas_sem_entrada += 1

    return {
        "total": total,
        "qtd_intervalos": qtd_intervalos,
        "entrada_aberta": entrada_aberta,
        "qtd_saidas_sem_entrada": qtd_saidas_sem_entrada,
        "teve_inconsistencia": (entrada_aberta is not None) or (qtd_saidas_sem_entrada > 0),
    }

def calcular_jornada_primeira_entrada_ultima_saida(grupo):
    """
    calcula a janela do dia como:
    ultima saida - primeira entrada
    """
    grupo = grupo.sort_values("data_marcacao").copy()

    entradas = grupo.loc[grupo["movimento"] == "entrada", "data_marcacao"]
    saidas = grupo.loc[grupo["movimento"] == "saida", "data_marcacao"]

    if entradas.empty or saidas.empty:
        return pd.NaT

    primeira_entrada = entradas.min()
    ultima_saida = saidas.max()

    if ultima_saida < primeira_entrada:
        return pd.NaT

    return ultima_saida - primeira_entrada

def achar_coluna(df, alvo):
    alvo = alvo.strip().lower()
    for c in df.columns:
        if str(c).strip().lower() == alvo:
            return c
    return None

# =========================================================
# leitura
# =========================================================
df = pd.read_parquet(arquivo_parquet).copy()

col_cadastro = achar_coluna(df, "cadastro")
col_nome = achar_coluna(df, "nome")
col_data = achar_coluna(df, "data")
col_hora = achar_coluna(df, "hora")
col_dir = achar_coluna(df, "dir")
col_desc = achar_coluna(df, "descricao coletor")
col_planta = achar_coluna(df, "planta")

colunas_necessarias = {
    "cadastro": col_cadastro,
    "nome": col_nome,
    "data": col_data,
    "hora": col_hora,
    "dir": col_dir,
    "descricao coletor": col_desc,
    "planta": col_planta,
}

faltantes = [k for k, v in colunas_necessarias.items() if v is None]
if faltantes:
    raise ValueError(f"colunas ausentes no parquet (em minusculo): {faltantes}")

df = df.rename(columns={
    col_cadastro: "chapa",
    col_nome: "nome",
    col_data: "data_str",
    col_hora: "hora_str",
    col_dir: "dir",
    col_desc: "descricao_coletor",
    col_planta: "planta",
})

df["chapa"] = df["chapa"].astype(str).str.strip()
df["nome"] = df["nome"].astype(str).str.strip()
df["dir"] = df["dir"].astype(str).str.strip().str.upper()
df["descricao_coletor"] = df["descricao_coletor"].astype(str)
df["planta"] = df["planta"].astype(str).str.strip()

# =========================================================
# datetime
# =========================================================
df["data_marcacao"] = pd.to_datetime(
    df["data_str"].astype(str).str.strip() + " " + df["hora_str"].astype(str).str.strip(),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)
df = df[df["data_marcacao"].notna()].copy()

dt_ini = pd.Timestamp(data_inicio)
dt_fim = pd.Timestamp(data_fim) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
df = df[(df["data_marcacao"] >= dt_ini) & (df["data_marcacao"] <= dt_fim)].copy()

# =========================================================
# filtro pinese (opcional)
# =========================================================
if chapas_pinese:
    chapas_set = {str(x).strip() for x in chapas_pinese}
    df = df[df["chapa"].isin(chapas_set)].copy()

# =========================================================
# movimento
# =========================================================
df["movimento"] = df["dir"].map({"E": "entrada", "S": "saida"})
df = df[df["movimento"].isin(["entrada", "saida"])].copy()
df["data"] = df["data_marcacao"].dt.date

# =========================================================
# REMOÇÃO DE DUPLICATAS
# regra: remover apenas duplicata exata da mesma batida
# =========================================================
df["hora"] = df["data_marcacao"].dt.strftime("%H:%M:%S")
df = df.sort_values(
    ["chapa", "nome", "data", "hora", "dir", "descricao_coletor", "planta", "data_marcacao"]
).copy()

df = df.drop_duplicates(
    subset=["nome", "chapa", "data", "hora", "dir", "descricao_coletor", "planta"],
    keep="first"
).copy()

# =========================================================
# base portaria para validação
# =========================================================
df_port = df[df["descricao_coletor"].str.contains("port", case=False, na=False)].copy()

# =========================================================
# validações de PORT
# =========================================================
registros_port_incongruentes = []

for (chapa, nome, data), grupo_port in df_port.groupby(["chapa", "nome", "data"], dropna=False):
    qtd_registros_port = len(grupo_port)
    qtd_entradas_port = int((grupo_port["movimento"] == "entrada").sum())
    qtd_saidas_port = int((grupo_port["movimento"] == "saida").sum())

    data_formatada = pd.to_datetime(data).strftime("%d/%m/%Y")
    dia_semana = dia_semana_ptbr(data)
    primeira_batida = grupo_port["data_marcacao"].min().strftime("%H:%M:%S") if not grupo_port.empty else None
    ultima_batida = grupo_port["data_marcacao"].max().strftime("%H:%M:%S") if not grupo_port.empty else None

    if qtd_entradas_port != qtd_saidas_port:
        registros_port_incongruentes.append({
            "chapa": chapa,
            "nome do colaborador": nome,
            "data": data_formatada,
            "dia da semana": dia_semana,
            "qtd_registros_port": qtd_registros_port,
            "qtd_entradas_port": qtd_entradas_port,
            "qtd_saidas_port": qtd_saidas_port,
            "primeira batida port": primeira_batida,
            "ultima batida port": ultima_batida,
            "status": "incongruência de entradas e saidas",
        })

df_port_incongruente = pd.DataFrame(registros_port_incongruentes)

# =========================================================
# filtro de cálculo
# =========================================================
if filtrar_somente_portaria:
    df_calc = df_port.copy()
else:
    df_calc = df.copy()

# =========================================================
# agrega: por colaborador + data
# =========================================================
registros = []
auditoria = []
group_cols = ["chapa", "nome", "data"]

for (chapa, nome, data), grupo in df_calc.groupby(group_cols, dropna=False):
    resultado = somar_intervalos_entrada_saida(grupo)

    total = resultado["total"]
    qtd_intervalos = resultado["qtd_intervalos"]
    entrada_aberta = resultado["entrada_aberta"]
    qtd_saidas_sem_entrada = resultado["qtd_saidas_sem_entrada"]
    teve_inconsistencia = resultado["teve_inconsistencia"]

    data_formatada = pd.to_datetime(data).strftime("%d/%m/%Y")
    dia_semana = dia_semana_ptbr(data)

    if teve_inconsistencia:
        status = []
        if entrada_aberta is not None:
            status.append("entrada sem saida")
        if qtd_saidas_sem_entrada > 0:
            status.append("saida sem entrada")

        auditoria.append({
            "chapa": chapa,
            "nome do colaborador": nome,
            "data": data_formatada,
            "dia da semana": dia_semana,
            "qtd_intervalos": qtd_intervalos,
            "horas calculadas": formatar_timedelta(total),
            "primeira batida": grupo["data_marcacao"].min().strftime("%H:%M:%S") if not grupo.empty else None,
            "ultima batida": grupo["data_marcacao"].max().strftime("%H:%M:%S") if not grupo.empty else None,
            "ultima entrada sem saida": entrada_aberta.strftime("%H:%M:%S") if pd.notna(entrada_aberta) else None,
            "qtd_saidas_sem_entrada": qtd_saidas_sem_entrada,
            "status": " / ".join(status),
        })
        continue

    if qtd_intervalos == 0:
        continue

    jornada_es = calcular_jornada_primeira_entrada_ultima_saida(grupo)

    registros.append({
        "chapa": chapa,
        "nome do colaborador": nome,
        "data": data_formatada,
        "dia da semana": dia_semana,
        "qtd_intervalos": qtd_intervalos,
        "horas na planta_td": total,
        "horas na planta": formatar_timedelta(total),
        "horas e-s_td": jornada_es,
        "horas e-s": formatar_timedelta(jornada_es),
    })

df_diario = pd.DataFrame(registros)
df_auditoria = pd.DataFrame(auditoria)

if (
    df_diario.empty
    and df_auditoria.empty
    and df_port_incongruente.empty
):
    raise ValueError(
        "nenhum intervalo válido entrada(e)->saida(s) encontrado no período. "
        "verifique se há pares e/s no mesmo dia."
    )

# =========================================================
# relatórios
# =========================================================
if not df_diario.empty:
    relatorio_acima_1050 = (
        df_diario[df_diario["horas na planta_td"] > limite_acima]
        .sort_values(["data", "nome do colaborador", "chapa"])
        .reset_index(drop=True)
    )

    relatorio_abaixo_0800 = (
        df_diario[df_diario["horas e-s_td"] < limite_abaixo]
        .sort_values(["data", "nome do colaborador", "chapa"])
        .reset_index(drop=True)
    )

    cols_saida_acima = [
        "chapa",
        "nome do colaborador",
        "data",
        "dia da semana",
        "qtd_intervalos",
        "horas na planta",
    ]
    cols_saida_abaixo = [
        "chapa",
        "nome do colaborador",
        "data",
        "dia da semana",
        "qtd_intervalos",
        "horas e-s",
    ]

    relatorio_acima_1050 = relatorio_acima_1050[cols_saida_acima]
    relatorio_abaixo_0800 = relatorio_abaixo_0800[cols_saida_abaixo]
else:
    relatorio_acima_1050 = pd.DataFrame(columns=[
        "chapa", "nome do colaborador", "data", "dia da semana", "qtd_intervalos", "horas na planta"
    ])
    relatorio_abaixo_0800 = pd.DataFrame(columns=[
        "chapa", "nome do colaborador", "data", "dia da semana", "qtd_intervalos", "horas e-s"
    ])

# =========================================================
# exportação - somente 3 arquivos
# =========================================================
arquivo_excel_acima = out_dir / "relatorio_acima_10h50.xlsx"
arquivo_excel_abaixo = out_dir / "relatorio_abaixo_08h00.xlsx"
arquivo_excel_port_incongruente = out_dir / "relatorio_dias_com_registros_de_port_e_s_incongruentes.xlsx"

with pd.ExcelWriter(arquivo_excel_acima, engine="openpyxl") as writer:
    relatorio_acima_1050.to_excel(writer, sheet_name="acima_10h50", index=False)

with pd.ExcelWriter(arquivo_excel_abaixo, engine="openpyxl") as writer:
    relatorio_abaixo_0800.to_excel(writer, sheet_name="abaixo_08h00", index=False)

with pd.ExcelWriter(arquivo_excel_port_incongruente, engine="openpyxl") as writer:
    df_port_incongruente.to_excel(writer, sheet_name="port_e_s_incongruentes", index=False)

print("relatórios gerados com sucesso:")
print(f"- {arquivo_excel_acima}")
print(f"- {arquivo_excel_abaixo}")
print(f"- {arquivo_excel_port_incongruente}")
print("\nresumo:")
print(f"- linhas acima de 10:50: {len(relatorio_acima_1050)}")
print(f"- linhas abaixo de 08:00: {len(relatorio_abaixo_0800)}")
print(f"- linhas com incongruência de entradas e saidas na portaria: {len(df_port_incongruente)}")

In [ ]:
# =========================================================
# CONFIG
# =========================================================
out_dir = Path("saida_relatorios_ronda")
arquivo_parquet = Path("df_ronda.parquet")
arquivo_saida = Path(r".\validations\checagem_3_exemplos_por_relatorio.xlsx")

relatorios = {
    "acima_10h50": out_dir / "relatorio_acima_10h50.xlsx",
    "abaixo_08h00": out_dir / "relatorio_abaixo_08h00.xlsx",
    "port_e_s_incongruentes": out_dir / "relatorio_dias_com_registros_de_port_e_s_incongruentes.xlsx",
}

# =========================================================
# HELPERS
# =========================================================
def achar_coluna(df, alvo):
    alvo = alvo.strip().lower()
    for c in df.columns:
        if str(c).strip().lower() == alvo:
            return c
    return None


def formatar_data_br(x):
    if pd.isna(x):
        return None
    try:
        return pd.to_datetime(x).strftime("%d/%m/%Y")
    except Exception:
        return str(x)


def auto_width(ws):
    for col_cells in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col_cells[0].column)
        for cell in col_cells:
            try:
                v = "" if cell.value is None else str(cell.value)
                if len(v) > max_len:
                    max_len = len(v)
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 40)


def normalizar_base(df_base):
    col_cadastro = achar_coluna(df_base, "cadastro")
    col_nome = achar_coluna(df_base, "nome")
    col_data = achar_coluna(df_base, "data")
    col_hora = achar_coluna(df_base, "hora")
    col_dir = achar_coluna(df_base, "dir")
    col_desc = achar_coluna(df_base, "descricao coletor")
    col_planta = achar_coluna(df_base, "planta")

    faltantes = [
        nome for nome, col in {
            "cadastro": col_cadastro,
            "nome": col_nome,
            "data": col_data,
            "hora": col_hora,
            "dir": col_dir,
            "descricao coletor": col_desc,
            "planta": col_planta,
        }.items() if col is None
    ]
    if faltantes:
        raise ValueError(f"Colunas ausentes na base original: {faltantes}")

    base = df_base.rename(columns={
        col_cadastro: "chapa",
        col_nome: "nome",
        col_data: "data_str",
        col_hora: "hora_str",
        col_dir: "dir",
        col_desc: "descricao_coletor",
        col_planta: "planta",
    }).copy()

    base["chapa"] = base["chapa"].astype(str).str.strip()
    base["nome"] = base["nome"].astype(str).str.strip()
    base["dir"] = base["dir"].astype(str).str.strip().str.upper()
    base["descricao_coletor"] = base["descricao_coletor"].astype(str).str.strip()
    base["planta"] = base["planta"].astype(str).str.strip()
    base["data_base"] = pd.to_datetime(base["data_str"], errors="coerce", dayfirst=True).dt.strftime("%d/%m/%Y")
    base["hora_base"] = base["hora_str"].astype(str).str.strip()
    base["data_marcacao"] = pd.to_datetime(
        base["data_str"].astype(str).str.strip() + " " + base["hora_str"].astype(str).str.strip(),
        errors="coerce",
        dayfirst=True,
    )
    base = base.sort_values(["chapa", "data_marcacao", "hora_base", "descricao_coletor"]).reset_index(drop=True)
    return base


# =========================================================
# CARGA
# =========================================================
if not arquivo_parquet.exists():
    raise FileNotFoundError(f"Não encontrei a base original: {arquivo_parquet.resolve()}")

for nome, arq in relatorios.items():
    if not arq.exists():
        raise FileNotFoundError(f"Não encontrei o relatório '{nome}': {arq.resolve()}")

base_original = normalizar_base(pd.read_parquet(arquivo_parquet))
relatorios_df = {nome: pd.read_excel(arq) for nome, arq in relatorios.items()}

# =========================================================
# EXCEL DE CHECAGEM
# =========================================================
wb = Workbook()
wb.remove(wb.active)

fill_titulo = PatternFill("solid", fgColor="1F4E78")
fill_header = PatternFill("solid", fgColor="D9EAF7")
fill_resumo = PatternFill("solid", fgColor="E2F0D9")
fill_base = PatternFill("solid", fgColor="FFF2CC")
fonte_branca = Font(color="FFFFFF", bold=True)
fonte_bold = Font(bold=True)
borda = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

for nome_relatorio, df_rel in relatorios_df.items():
    ws = wb.create_sheet(title=nome_relatorio[:31])
    linha = 1

    ws.cell(linha, 1).value = f"Checagem - {nome_relatorio}"
    ws.cell(linha, 1).fill = fill_titulo
    ws.cell(linha, 1).font = fonte_branca
    ws.cell(linha, 1).alignment = Alignment(horizontal="left")
    linha += 2

    exemplos = df_rel.head(3).copy()

    if exemplos.empty:
        ws.cell(linha, 1).value = "Sem registros neste relatório."
        continue

    for idx, ex in exemplos.reset_index(drop=True).iterrows():
        exemplo_num = idx + 1
        chapa = str(ex.get("chapa", "")).strip()
        data_ref = str(ex.get("data", "")).strip()

        ws.cell(linha, 1).value = f"Exemplo {exemplo_num}"
        ws.cell(linha, 1).font = fonte_bold
        linha += 1

        # Resumo do relatório
        resumo_cols = list(exemplos.columns)
        for col_idx, col_name in enumerate(resumo_cols, start=1):
            c = ws.cell(linha, col_idx)
            c.value = col_name
            c.fill = fill_header
            c.font = fonte_bold
            c.border = borda
        linha += 1

        for col_idx, col_name in enumerate(resumo_cols, start=1):
            c = ws.cell(linha, col_idx)
            c.value = ex.get(col_name)
            c.fill = fill_resumo
            c.border = borda
        linha += 2

        # Base original do mesmo colaborador/dia
        detalhe = base_original[
            (base_original["chapa"] == chapa) &
            (base_original["data_base"] == data_ref)
        ].copy()

        detalhe_cols = [
            "chapa", "nome", "data_base", "hora_base", "dir",
            "planta", "descricao_coletor", "data_marcacao"
        ]

        ws.cell(linha, 1).value = "Linhas da base original para comparação"
        ws.cell(linha, 1).font = fonte_bold
        linha += 1

        for col_idx, col_name in enumerate(detalhe_cols, start=1):
            c = ws.cell(linha, col_idx)
            c.value = col_name
            c.fill = fill_header
            c.font = fonte_bold
            c.border = borda
        linha += 1

        if detalhe.empty:
            ws.cell(linha, 1).value = "Nenhuma linha encontrada na base original para este exemplo."
            linha += 2
        else:
            for _, row in detalhe.iterrows():
                for col_idx, col_name in enumerate(detalhe_cols, start=1):
                    val = row.get(col_name)
                    if col_name == "data_marcacao" and pd.notna(val):
                        val = pd.to_datetime(val).strftime("%d/%m/%Y %H:%M:%S")
                    c = ws.cell(linha, col_idx)
                    c.value = val
                    c.fill = fill_base
                    c.border = borda
                linha += 1
            linha += 1

    auto_width(ws)
    ws.freeze_panes = "A3"

wb.save(arquivo_saida)
print(f"Arquivo gerado com sucesso: {arquivo_saida}")